# MCQ to OSQ Conversion

This notebook demonstrates converting multiple-choice questions (MCQ) to open-style questions (OSQ).

In [ ]:
import os
import pandas as pd

from metaeval.benchmark.convert import MCQToOSQConverter, conversions_to_dataframe
from metaeval.prompts import get_registry, get_prompt

## Setup API Key

Conversion requires an OpenAI API key:

In [ ]:
# Set your API key (or use environment variable)
# os.environ['OPENAI_API_KEY'] = 'your-key-here'

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print("API key found!")
else:
    print("Warning: OPENAI_API_KEY not set. Conversion will fail.")

## Sample MCQ Data

In [ ]:
# Create sample MCQ data
mcq_data = pd.DataFrame({
    'question_id': ['q1', 'q2', 'q3'],
    'question': [
        'What is the primary purpose of systems engineering?',
        'Which methodology emphasizes iterative development and customer collaboration?',
        'What does INCOSE stand for?',
    ],
    'choice_a': [
        'To write code efficiently',
        'Waterfall',
        'International Network of Computer Operating System Engineers',
    ],
    'choice_b': [
        'To manage complex system development holistically',
        'Agile',
        'International Council on Systems Engineering',
    ],
    'choice_c': [
        'To test software applications',
        'CMMI',
        'Institute for Network and Computer Science Engineering',
    ],
    'choice_d': [
        'To design user interfaces',
        'Six Sigma',
        'International Committee on Software Engineering',
    ],
    'answer': ['B', 'B', 'B'],
    'justification': [
        'Systems engineering takes a holistic approach to complex system development.',
        'Agile methodology emphasizes iterative development, flexibility, and customer collaboration.',
        'INCOSE is the International Council on Systems Engineering.',
    ],
})

mcq_data

## View Available Conversion Prompts

In [ ]:
# List available conversion prompts
registry = get_registry()
convert_prompts = registry.list('convert')

print("Available Conversion Prompts:")
for prompt in convert_prompts:
    print(f"  - {prompt.name}: {prompt.description.split(chr(10))[0]}")

In [ ]:
# View a specific prompt template
standard_prompt = get_prompt('standard', 'convert')
if standard_prompt:
    print(f"Prompt: {standard_prompt.name}")
    print(f"Variables: {standard_prompt.variables}")
    print("\nTemplate (first 500 chars):")
    print(standard_prompt.template[:500])

## Configure Converter

In [ ]:
# Create converter with custom settings
converter = MCQToOSQConverter(
    api_key=api_key or 'placeholder',  # Will fail without real key
    model='gpt-4o',
    confidence_threshold=7,  # Only convert if suitability >= 7
    classification_prompt='classification',
    conversion_prompt='standard',
)

print(f"Converter configured:")
print(f"  Model: {converter.model}")
print(f"  Threshold: {converter.confidence_threshold}")

## Run Conversion

Note: This requires a valid API key to work.

In [ ]:
# Batch convert (requires API key)
if api_key:
    conversions, stats = converter.batch_convert(mcq_data)
    
    print(f"Converted {len(conversions)} questions")
    print(f"Conversion stats: {stats}")
    
    # View results
    result_df = conversions_to_dataframe(conversions)
    display(result_df[['question_id', 'osq_prompt', 'blooms_level', 'suitability_score']])
else:
    print("Skipping conversion - no API key")
    print("\nExample output would look like:")
    example = {
        'question_id': 'q1',
        'osq_prompt': 'Explain the primary purpose and role of systems engineering in managing complex projects.',
        'expected_answer': 'Systems engineering takes a holistic approach to developing complex systems...',
        'rubric': {
            'full_credit': 'Mentions holistic approach, complexity management, integration',
            'partial_credit': 'Addresses some key concepts but incomplete',
            'no_credit': 'Incorrect or irrelevant response',
        },
        'blooms_level': 'Understand',
        'suitability_score': 9,
    }
    for k, v in example.items():
        print(f"{k}: {v}")

## Understanding the Output

Each converted question includes:

- **osq_prompt**: The open-ended question version
- **expected_answer**: Model answer for judging
- **rubric**: Grading criteria (full/partial/no credit)
- **blooms_level**: Bloom's taxonomy classification
- **suitability_score**: How suitable the question is for OSQ format (1-10)

## CLI Alternative

You can also convert from the command line:

```bash
# Basic conversion
metaeval convert questions.csv -o converted.csv

# With custom settings
metaeval convert questions.csv \
    --model gpt-4-turbo \
    --threshold 8 \
    --prompt standard

# List available prompts
metaeval prompts convert
```